### Library Imports
This cell imports all required libraries for computer vision, feature extraction, and machine learning.




In [14]:
import os
import cv2
import numpy as np
import pywt
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


###  Feature Extraction
Extracting image features using SIFT, DWT and multi-scale LBP.


In [11]:
def extract_combined_features(image_path):
    """
    Extracts unified feature vectors using SIFT (Statistical Descriptors), DWT, and LBP.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 1. SIFT Features (Fixed-size Statistical Descriptor)
    sift = cv2.SIFT_create()
    keypoints, descriptors = sift.detectAndCompute(blurred, None)
    if descriptors is not None and len(descriptors) > 0:
        sift_feats = [
            np.mean(descriptors), np.std(descriptors),
            np.max(descriptors), np.min(descriptors),
            len(keypoints)
        ]
    else:
        sift_feats = [0, 0, 0, 0, 0]
    
    # 2. DWT Features
    coeffs = pywt.dwt2(blurred, 'haar')
    LL, (LH, HL, HH) = coeffs
    dwt_feats = [
        np.mean(LL), np.std(LL), np.var(LL),
        np.mean(LH), np.std(LH),
        np.mean(HL), np.std(HL),
        np.mean(HH), np.std(HH)
    ]
    
    # 3. LBP Features
    lbp1 = local_binary_pattern(blurred, P=8, R=1, method="uniform")
    hist1, _ = np.histogram(lbp1.ravel(), bins=np.arange(0, 11), density=True)
    
    lbp2 = local_binary_pattern(blurred, P=16, R=2, method="uniform")
    hist2, _ = np.histogram(lbp2.ravel(), bins=np.arange(0, 19), density=True)
    
    return np.hstack([sift_feats, dwt_feats, hist1, hist2])



### Dataset Loading
Loading images, assigning labels (Authentic/Forged), and creating the feature matrix X and label vector y.


In [12]:
dataset_dir = r"C:\Users\USER\Desktop\CMFD_Project\dataset\MICC-F220"

X = []  # Feature vectors matrix
y = []  # Labels: 0 for Authentic, 1 for Forged

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')

if os.path.exists(dataset_dir):
    image_files = [f for f in os.listdir(dataset_dir) if f.lower().endswith(valid_extensions)]
    
    for filename in image_files:
        path = os.path.join(dataset_dir, filename)
        name_lower = filename.lower()
        
        # Determine label based on filename logic:
        # Authentic images end with '_authentic'
        # Forged images contain 'tamp', '_scale', or '_rot'
        if '_authentic' in name_lower:
            label = 0  # Authentic (Healthy)
        elif any(k in name_lower for k in ['tamp', '_scale', '_rot']):
            label = 1  # Forged (Copy-Move)
        else:
            continue  # Skip any unrecognized files
            
        vec = extract_combined_features(path)
        if vec is not None:
            X.append(vec)
            y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Dataset successfully loaded!")
print(f"Total samples: {X.shape[0]} | Feature vector dimension: {X.shape[1] if X.ndim > 1 else 0}")
print(f"Authentic samples (Label 0): {np.sum(y == 0)}")
print(f"Forged samples (Label 1)   : {np.sum(y == 1)}")


Dataset successfully loaded!
Total samples: 326 | Feature vector dimension: 42
Authentic samples (Label 0): 106
Forged samples (Label 1)   : 220


### Model Training & Evaluation
Splitting the data into 80% training and 20% testing, scaling the features with StandardScaler, training a balanced SVM model, and evaluating its performance.


In [15]:
# 1. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train Support Vector Classifier
svm_classifier = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced', random_state=42)
print("Training Optimized Texture-based SVM Classifier...")
svm_classifier.fit(X_train_scaled, y_train)

# 4. Predictions
y_pred = svm_classifier.predict(X_test_scaled)

# 5. Output Results
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n================ Classification Results ================")
print(f"Model Accuracy: {acc * 100:.2f}%\n")
print("Confusion Matrix:")
print(cm)
print("\nDetailed Performance Metrics:")
print(classification_report(y_test, y_pred, target_names=['Authentic (0)', 'Forged (1)']))



Training Optimized Texture-based SVM Classifier...

================ Classification Results ================
Model Accuracy: 90.91%

Confusion Matrix:
[[17  4]
 [ 2 43]]

Detailed Performance Metrics:
               precision    recall  f1-score   support

Authentic (0)       0.89      0.81      0.85        21
   Forged (1)       0.91      0.96      0.93        45

     accuracy                           0.91        66
    macro avg       0.90      0.88      0.89        66
 weighted avg       0.91      0.91      0.91        66

